# Task 3: Transformer-Based Modeling

This notebook implements the required Transformer-based modeling section using a pretrained BERT model.

The three strategies implemented are:

1. **Strategy 1 — Domain Only Fine-Tuning**
2. **Strategy 2 — Sequential Transfer Learning**
3. **Strategy 3 — Mixed Training**

The models are compared quantitatively using Accuracy, Precision, Recall, and F1-score.


## Cell 1: Install Required Libraries

**Why we use this:**  
These libraries are needed for BERT, PyTorch training, data handling, and model evaluation.


In [68]:
# !pip install transformers torch scikit-learn pandas numpy

## Cell 2: Import Libraries

**Why we use this:**  
These imports allow us to load datasets, tokenize text, train BERT, and evaluate performance.


In [69]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader


## Cell 3: Load the Three Datasets

**Dataset usage:**

- `goemotions_5class.csv` = General dataset
- `fpb_5class.csv` = Domain dataset
- `combined_5class.csv` = Mixed dataset

**Why we use this:**  
The assignment requires comparing general/domain/mixed training strategies.


In [70]:
general_df = pd.read_csv("goemotions_5class.csv")
domain_df = pd.read_csv("fpb_5class.csv")
combined_df = pd.read_csv("combined_5class.csv")

print("General dataset shape:", general_df.shape)
print("Domain dataset shape:", domain_df.shape)
print("Combined dataset shape:", combined_df.shape)

display(general_df.head())
display(domain_df.head())
display(combined_df.head())


General dataset shape: (43404, 3)
Domain dataset shape: (4846, 3)
Combined dataset shape: (48250, 3)


,text,label,source
0,my favourite food is anything i didn't have to...,Neutral,GoEmotions
1,"now if he does off himself, everyone will thin...",Neutral,GoEmotions
2,why the fuck is bayless isoing,Fear,GoEmotions
3,to make her feel threatened,Fear,GoEmotions
4,dirty southern wankers,Fear,GoEmotions


,text,label,source
0,"according to gran , the company has no plans t...",Neutral,FinancialPhraseBank
1,technopolis plans to develop in stages an area...,Neutral,FinancialPhraseBank
2,the international electronic industry company ...,Sadness,FinancialPhraseBank
3,with the new production plant the company woul...,Optimism,FinancialPhraseBank
4,according to the company 's updated strategy f...,Optimism,FinancialPhraseBank


,text,label,source
0,my favourite food is anything i didn't have to...,Neutral,GoEmotions
1,"now if he does off himself, everyone will thin...",Neutral,GoEmotions
2,why the fuck is bayless isoing,Fear,GoEmotions
3,to make her feel threatened,Fear,GoEmotions
4,dirty southern wankers,Fear,GoEmotions


## Cell 4: Check Column Names

**Why we use this:**  
Before training, we must confirm the text and label columns in each dataset.


In [71]:
print("General columns:", general_df.columns.tolist())
print("Domain columns:", domain_df.columns.tolist())
print("Combined columns:", combined_df.columns.tolist())


General columns: ['text', 'label', 'source']
Domain columns: ['text', 'label', 'source']
Combined columns: ['text', 'label', 'source']


## Cell 5: Set Text and Label Columns

**Important:**  
If your CSV files use different names, change only these two variables.

Common possibilities:
- text column: `text`, `sentence`, `review`
- label column: `label`, `sentiment`, `class`


In [72]:
text_column = "text"
label_column = "label"


## Cell 6: Clean Dataset Rows

**Why we use this:**  
This removes missing values and makes sure the text is stored as string format.


In [73]:
general_df = general_df[[text_column, label_column]].dropna()
domain_df = domain_df[[text_column, label_column]].dropna()
combined_df = combined_df[[text_column, label_column]].dropna()

general_df[text_column] = general_df[text_column].astype(str)
domain_df[text_column] = domain_df[text_column].astype(str)
combined_df[text_column] = combined_df[text_column].astype(str)

print("General dataset after cleaning:", general_df.shape)
print("Domain dataset after cleaning:", domain_df.shape)
print("Combined dataset after cleaning:", combined_df.shape)


General dataset after cleaning: (43404, 2)
Domain dataset after cleaning: (4846, 2)
Combined dataset after cleaning: (48250, 2)


## Cell 7: Check Label Values

**Why we use this:**  
We need to know whether the labels are text labels or already numeric.


In [74]:
print("General labels:", general_df[label_column].unique())
print("Domain labels:", domain_df[label_column].unique())
print("Combined labels:", combined_df[label_column].unique())


General labels: <StringArray>
['Neutral', 'Fear', 'Joy', 'Optimism', 'Sadness']
Length: 5, dtype: str
Domain labels: <StringArray>
['Neutral', 'Sadness', 'Optimism', 'Joy', 'Fear']
Length: 5, dtype: str
Combined labels: <StringArray>
['Neutral', 'Fear', 'Joy', 'Optimism', 'Sadness']
Length: 5, dtype: str


## Cell 8: Convert Labels to Numbers

**Why we use this:**  
BERT needs numeric class labels for training.

This code supports both:
- text labels such as `negative`, `neutral`, `positive`
- numeric labels such as `0`, `1`, `2`, `3`, `4`


In [ ]:
label_map = {
    "neutral": 0,
    "optimism": 1,
    "sadness": 2,
    "joy": 3,
    "fear": 4
}

def convert_labels(df, label_column):
    # If labels are already numeric, keep them as integers
    if pd.api.types.is_numeric_dtype(df[label_column]):
        df[label_column] = df[label_column].astype(int)
        return df

    # Otherwise map text labels to numbers
    df[label_column] = df[label_column].astype(str).str.lower().str.strip()
    df[label_column] = df[label_column].map(label_map)
    df = df.dropna()
    df[label_column] = df[label_column].astype(int)
    return df

domain_df = convert_labels(domain_df, label_column)
combined_df = convert_labels(combined_df, label_column)

print("General final shape:", general_df.shape)
print("Domain final shape:", domain_df.shape)
print("Combined final shape:", combined_df.shape)

print("General label counts:")
print(general_df[label_column].value_counts())

print("\nDomain label counts:")
print(domain_df[label_column].value_counts())

print("\nCombined label counts:")
print(combined_df[label_column].value_counts())


General final shape: (43404, 2)
Domain final shape: (4846, 2)
Combined final shape: (48250, 2)
General label counts:
label
0    17312
1     9302
3     7624
4     6520
2     2646
Name: count, dtype: int64

Domain label counts:
label
0    2879
1    1303
2     558
3      60
4      46
Name: count, dtype: int64

Combined label counts:
label
0    20191
1    10605
3     7684
4     6566
2     3204
Name: count, dtype: int64


## Cell 9: Load BERT Tokenizer

**Why we use this:**  
BERT cannot read raw text directly. The tokenizer converts text into BERT input format.


In [76]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


## Cell 10: Create Dataset Class

**Why we use this:**  
This class prepares each text sample as:
- input IDs
- attention mask
- label

These are the inputs required by BERT.


In [77]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = str(self.texts[index])
        label = int(self.labels[index])

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "label": torch.tensor(label, dtype=torch.long)
        }


## Cell 11: Training Function

**Why we use this:**  
This function performs one full training pass through the dataset.


In [78]:
def train_model(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0

    for batch in dataloader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    return total_loss / len(dataloader)


## Cell 12: Evaluation Function

**Why we use this:**  
The assignment requires quantitative comparison. This function calculates Accuracy, Precision, Recall, and F1-score.


In [ ]:
def evaluate_model(model, dataloader, device):
    model.eval()

    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = torch.argmax(outputs.logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(true_labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)
    print("\nClassification Report:\n")
    print(classification_report(true_labels, predictions, zero_division=0))
    # print("True labels unique:", set(true_labels))
    # print("Predictions unique:", set(predictions))
    # print("Num true labels:", len(true_labels))
    # print("Num predictions:", len(predictions))
    # print(np.unique(true_labels, return_counts=True))
    # print(np.unique(predictions, return_counts=True))
    print("\nConfusion Matrix:\n")
    print(confusion_matrix(true_labels, predictions))

    return accuracy, precision, recall, f1


## Cell 13: Set Device and Hyperparameters

**Why we use this:**  
BERT trains faster on GPU. These hyperparameters match the design concept.


In [80]:
# Set MPS as default device
import torch

# Check MPS availability
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS device")
else:
    device = torch.device("cpu")
    print("MPS not available, using CPU")

# Set default tensor type for MPS
if device.type == "mps":
    torch.set_default_tensor_type('torch.FloatTensor')

MAX_LEN = 128
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
EPOCHS = 3
NUM_LABELS = 5

print("Using device:", device)


Using MPS device
Using device: mps


# Strategy 1 — Domain Only Fine-Tuning

Train BERT only on the domain dataset.

**Why we use this:**  
This tests how well BERT performs when trained only on domain-specific data.


In [81]:
domain_train_texts, domain_val_texts, domain_train_labels, domain_val_labels = train_test_split(
    domain_df[text_column].values,
    domain_df[label_column].values,
    test_size=0.2,
    random_state=42,
    stratify=domain_df[label_column].values
)

domain_train_dataset = SentimentDataset(domain_train_texts, domain_train_labels, tokenizer, MAX_LEN)
domain_val_dataset = SentimentDataset(domain_val_texts, domain_val_labels, tokenizer, MAX_LEN)

domain_train_loader = DataLoader(domain_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
domain_val_loader = DataLoader(domain_val_dataset, batch_size=BATCH_SIZE)

print("Domain training samples:", len(domain_train_dataset))
print("Domain validation samples:", len(domain_val_dataset))


Domain training samples: 3876
Domain validation samples: 970


In [82]:
model_domain = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=NUM_LABELS
)

model_domain.to(device)

optimizer = AdamW(model_domain.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    train_loss = train_model(model_domain, domain_train_loader, optimizer, device)
    print("Training Loss:", train_loss)
    domain_only_results = evaluate_model(model_domain, domain_val_loader, device)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7738.93it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

Epoch 1/3
Training Loss: 0.7665939981064188
Accuracy: 0.8340206185567011
Precision: 0.8153981049789684
Recall: 0.8340206185567011
F1 Score: 0.8199117932696688

Classification Report:

              precision    recall  f1-score   support

           0       0.85      0.95      0.89       576
           1       0.83      0.65      0.73       261
           2       0.77      0.85      0.81       112
           3       0.00      0.00      0.00        12
           4       0.00      0.00      0.00         9

    accuracy                           0.83       970
   macro avg       0.49      0.49      0.49       970
weighted avg       0.82      0.83      0.82       970

True labels unique: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)}
Predictions unique: {np.int64(0), np.int64(1), np.int64(2)}
Num true labels: 970
Num predictions: 970
(array([0, 1, 2, 3, 4]), array([576, 261, 112,  12,   9]))
(array([0, 1, 2]), array([643, 204, 123]))

Confusion Matrix:

[[545  16  15   0

# Strategy 2 — Sequential Transfer Learning

Train on the general dataset first, then fine-tune on the domain dataset.

**Why we use this:**  
This is the main transfer learning experiment. The model first learns general sentiment patterns, then adapts to domain-specific language.


In [83]:
general_train_texts, general_val_texts, general_train_labels, general_val_labels = train_test_split(
    general_df[text_column].values,
    general_df[label_column].values,
    test_size=0.2,
    random_state=42,
    stratify=general_df[label_column].values
)

general_train_dataset = SentimentDataset(general_train_texts, general_train_labels, tokenizer, MAX_LEN)
general_train_loader = DataLoader(general_train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print("General training samples:", len(general_train_dataset))


General training samples: 34723


In [85]:
model_sequential = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=NUM_LABELS
)

model_sequential.to(device)

optimizer = AdamW(model_sequential.parameters(), lr=LEARNING_RATE)

print("Stage 1: Training on general dataset")

for epoch in range(EPOCHS):
    print(f"General Training Epoch {epoch + 1}/{EPOCHS}")
    train_loss = train_model(model_sequential, general_train_loader, optimizer, device)
    print("General Training Loss:", train_loss)
    


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8146.03it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

Stage 1: Training on general dataset
General Training Epoch 1/3
General Training Loss: 0.8763807185409477
General Training Epoch 2/3
General Training Loss: 0.6650983709335766
General Training Epoch 3/3
General Training Loss: 0.4717325691712007


In [86]:
optimizer = AdamW(model_sequential.parameters(), lr=LEARNING_RATE)

print("Stage 2: Fine-tuning on domain dataset")

for epoch in range(EPOCHS):
    print(f"Domain Fine-Tuning Epoch {epoch + 1}/{EPOCHS}")
    train_loss = train_model(model_sequential, domain_train_loader, optimizer, device)
    print("Domain Fine-Tuning Loss:", train_loss)
    sequential_results = evaluate_model(model_sequential, domain_val_loader, device)


Stage 2: Fine-tuning on domain dataset
Domain Fine-Tuning Epoch 1/3
Domain Fine-Tuning Loss: 0.5032146983615164
Accuracy: 0.8360824742268042
Precision: 0.8368902710977123
Recall: 0.8360824742268042
F1 Score: 0.8342963462444629

Classification Report:

              precision    recall  f1-score   support

           0       0.90      0.87      0.88       576
           1       0.72      0.83      0.77       261
           2       0.87      0.79      0.83       112
           3       0.43      0.25      0.32        12
           4       0.33      0.11      0.17         9

    accuracy                           0.84       970
   macro avg       0.65      0.57      0.59       970
weighted avg       0.84      0.84      0.83       970

True labels unique: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)}
Predictions unique: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)}
Num true labels: 970
Num predictions: 970
(array([0, 1, 2, 3, 4]), array([576, 261, 112

# Strategy 3 — Mixed Training

Train BERT once using the combined dataset.

**Why we use this:**  
This tests whether combining general and domain data gives more stable performance.


In [87]:
combined_train_texts, combined_val_texts, combined_train_labels, combined_val_labels = train_test_split(
    combined_df[text_column].values,
    combined_df[label_column].values,
    test_size=0.2,
    random_state=42,
    stratify=combined_df[label_column].values
)

combined_train_dataset = SentimentDataset(combined_train_texts, combined_train_labels, tokenizer, MAX_LEN)
combined_val_dataset = SentimentDataset(combined_val_texts, combined_val_labels, tokenizer, MAX_LEN)

combined_train_loader = DataLoader(combined_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
combined_val_loader = DataLoader(combined_val_dataset, batch_size=BATCH_SIZE)

print("Combined training samples:", len(combined_train_dataset))
print("Combined validation samples:", len(combined_val_dataset))


Combined training samples: 38600
Combined validation samples: 9650


In [88]:
model_mixed = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=NUM_LABELS
)

model_mixed.to(device)

optimizer = AdamW(model_mixed.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    print(f"Mixed Training Epoch {epoch + 1}/{EPOCHS}")
    train_loss = train_model(model_mixed, combined_train_loader, optimizer, device)
    print("Mixed Training Loss:", train_loss)
    mixed_results = evaluate_model(model_mixed, combined_val_loader, device)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8990.67it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

Mixed Training Epoch 1/3
Mixed Training Loss: 0.832143371709392
Accuracy: 0.7179274611398964
Precision: 0.7165703085437325
Recall: 0.7179274611398964
F1 Score: 0.7167202944490103

Classification Report:

              precision    recall  f1-score   support

           0       0.76      0.76      0.76      4038
           1       0.67      0.69      0.68      2121
           2       0.67      0.58      0.62       641
           3       0.75      0.81      0.78      1537
           4       0.64      0.59      0.62      1313

    accuracy                           0.72      9650
   macro avg       0.70      0.69      0.69      9650
weighted avg       0.72      0.72      0.72      9650

True labels unique: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)}
Predictions unique: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)}
Num true labels: 9650
Num predictions: 9650
(array([0, 1, 2, 3, 4]), array([4038, 2121,  641, 1537, 1313]))
(array([0, 1, 2, 3, 4]), ar

# Final Quantitative Comparison

All models are compared on the **same domain validation dataset**.

**Why we use this:**  
The project requires comparing the Transformer strategies quantitatively. Using the same domain validation set makes the comparison fair.


In [89]:
print("Final Evaluation: Domain Only Model")
domain_only_results = evaluate_model(model_domain, domain_val_loader, device)

print("\nFinal Evaluation: Sequential Transfer Learning Model")
sequential_results = evaluate_model(model_sequential, domain_val_loader, device)

print("\nFinal Evaluation: Mixed Training Model")
mixed_results_on_domain = evaluate_model(model_mixed, domain_val_loader, device)


Final Evaluation: Domain Only Model
Accuracy: 0.8546391752577319
Precision: 0.8478985601103771
Recall: 0.8546391752577319
F1 Score: 0.8489087870535842

Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.94      0.90       576
           1       0.85      0.75      0.80       261
           2       0.82      0.79      0.81       112
           3       0.25      0.17      0.20        12
           4       0.33      0.11      0.17         9

    accuracy                           0.85       970
   macro avg       0.63      0.55      0.58       970
weighted avg       0.85      0.85      0.85       970

True labels unique: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)}
Predictions unique: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)}
Num true labels: 970
Num predictions: 970
(array([0, 1, 2, 3, 4]), array([576, 261, 112,  12,   9]))
(array([0, 1, 2, 3, 4]), array([622, 229, 108,   8,   3]))

Co

In [90]:
results_df = pd.DataFrame({
    "Strategy": [
        "Strategy 1 - Domain Only Fine-Tuning",
        "Strategy 2 - Sequential Transfer Learning",
        "Strategy 3 - Mixed Training"
    ],
    "Accuracy": [
        domain_only_results[0],
        sequential_results[0],
        mixed_results_on_domain[0]
    ],
    "Precision": [
        domain_only_results[1],
        sequential_results[1],
        mixed_results_on_domain[1]
    ],
    "Recall": [
        domain_only_results[2],
        sequential_results[2],
        mixed_results_on_domain[2]
    ],
    "F1 Score": [
        domain_only_results[3],
        sequential_results[3],
        mixed_results_on_domain[3]
    ]
})

results_df


,Strategy,Accuracy,Precision,Recall,F1 Score
0,Strategy 1 - Domain Only Fine-Tuning,0.854639,0.847899,0.854639,0.848909
1,Strategy 2 - Sequential Transfer Learning,0.834021,0.831807,0.834021,0.832359
2,Strategy 3 - Mixed Training,0.957732,0.957959,0.957732,0.957655


## Save the Best Model

**Why we use this:**  
The best model can be reused later without training again.


In [91]:
best_model = model_sequential

best_model.save_pretrained("best_bert_task3_model")
tokenizer.save_pretrained("best_bert_task3_model")

print("Best model saved successfully.")


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.26it/s]

Best model saved successfully.


## Short Report Explanation

For Task 3, a pretrained BERT model was used for Transformer-based sentiment classification. Three training strategies were implemented and compared. The first strategy fine-tuned BERT only on the domain dataset. The second strategy used sequential transfer learning by first training BERT on a general sentiment dataset and then fine-tuning it on the domain dataset. The third strategy trained BERT once on a combined dataset containing both general and domain samples. Each strategy was evaluated using accuracy, precision, recall, and F1-score to determine how training data choice influenced domain-specific sentiment classification performance.
